# 📥 Pipeline d'Ingestion Multi-Format — Study-Buddy

**Projet SMA Agentic AI — EMSI**  
**Étudiantes :** EL OUALFI Meriem & BOUTARFA Wafae  

---

Ce notebook documente la **pipeline d'ingestion RAG** du projet Study-Buddy.  
Il montre comment un fichier (PDF, PPTX, DOCX, TXT, image) est traité, vectorisé et stocké dans Pinecone.

## Architecture d'ingestion

```
Fichier Upload
     │
     ▼
Webhook n8n (rag-ingestion)
     │
     ▼
Détecter Type Fichier (Switch)
     │
     ├── PDF  → Extraire texte (PyPDF)
     ├── PPTX → Extraire texte (python-pptx)
     ├── DOCX → Extraire texte (python-docx)
     ├── TXT  → Lecture directe
     └── IMG  → OCR via Gemini Vision
          │
          ▼
     Normaliser + Nettoyer texte
          │
          ▼
     Text Splitter (chunks 500 tokens / overlap 50)
          │
          ▼
     Embeddings Google Gemini
          │
          ▼
     Pinecone — Stocker Vecteurs
     (Index: studybuddy-index | Namespace: fileName)
```


## 1. Configuration

In [ ]:
import requests
import os
import json
import time

# URLs des webhooks n8n
N8N_UPLOAD_URL = "https://sttudybuddy.app.n8n.cloud/webhook/rag-ingestion"

# Configuration Pinecone (pour documentation)
PINECONE_INDEX = "studybuddy-index"
EMBEDDING_MODEL = "Google Gemini Embeddings"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

print("✅ Configuration chargée")
print(f"   Index Pinecone : {PINECONE_INDEX}")
print(f"   Modèle embedding : {EMBEDDING_MODEL}")
print(f"   Taille des chunks : {CHUNK_SIZE} tokens")
print(f"   Overlap : {CHUNK_OVERLAP} tokens")

✅ Configuration chargée
   Index Pinecone : studybuddy-index
   Modèle embedding : Google Gemini Embeddings
   Taille des chunks : 500 tokens
   Overlap : 50 tokens


## 2. Stratégie de Chunking

Nous utilisons un **RecursiveCharacterTextSplitter** avec :
- `chunk_size = 500` tokens — équilibre entre contexte et précision
- `chunk_overlap = 50` tokens — évite de couper les phrases importantes

**Pourquoi ces valeurs ?**  
Un chunk trop grand → perd la précision de la recherche  
Un chunk trop petit → perd le contexte sémantique  
500 tokens est le standard recommandé pour les cours académiques.

In [ ]:
def simulate_chunking(text, chunk_size=500, overlap=50):
    """
    Simule le découpage de texte en chunks.
    En production, cette opération est faite par n8n (TextSplitterRecursiveCharacter).
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# Exemple de démonstration
exemple_texte = """
Le RAG (Retrieval-Augmented Generation) est une architecture qui combine la recherche
d'information et la génération de texte. Elle permet à un LLM d'accéder à des
connaissances externes sans avoir besoin d'être réentraîné. Les embeddings sont des
vecteurs numériques qui représentent le sens sémantique d'un texte dans un espace
mathématique de haute dimension.
"""

chunks = simulate_chunking(exemple_texte, chunk_size=20, overlap=5)
print(f"✅ Texte découpé en {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"   Chunk {i+1}: {chunk[:80]}..." if len(chunk) > 80 else f"   Chunk {i+1}: {chunk}")

✅ Texte découpé en 4 chunks
   Chunk 1: Le RAG (Retrieval-Augmented Generation) est une architecture qui combine la rech...
   Chunk 2: de texte. Elle permet à un LLM d'accéder à des connaissances externes sans avoir...
   Chunk 3: d'être réentraîné. Les embeddings sont des vecteurs numériques qui représentent ...
   Chunk 4: dans un espace mathématique de haute dimension.


## 3. Test d'ingestion — Upload d'un fichier réel

In [ ]:
import requests
import os
import json
import time
from google.colab import files

# URLs des webhooks n8n
N8N_UPLOAD_URL = "https://sttudybuddy.app.n8n.cloud/webhook/rag-ingestion"

# Configuration Pinecone (pour documentation)
PINECONE_INDEX = "studybuddy-index"
EMBEDDING_MODEL = "Google Gemini Embeddings"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

print("✅ Configuration chargée")
print(f"   Index Pinecone : {PINECONE_INDEX}")
print(f"   Modèle embedding : {EMBEDDING_MODEL}")
print(f"   Taille des chunks : {CHUNK_SIZE} tokens")
print(f"   Overlap : {CHUNK_OVERLAP} tokens")

def upload_fichier(chemin_fichier):
    """
    Envoie un fichier au webhook n8n pour ingestion dans Pinecone.

    Args:
        chemin_fichier: chemin local vers le fichier à uploader

    Returns:
        dict: réponse du serveur avec détails de l'ingestion
    """
    nom_fichier = os.path.basename(chemin_fichier)

    print(f"📤 Upload de : {nom_fichier}")
    print(f"   Vers : {N8N_UPLOAD_URL}")

    with open(chemin_fichier, 'rb') as f:
        files_data = {'file': (nom_fichier, f, 'application/octet-stream')}
        data = {'fileName': nom_fichier}

        debut = time.time()
        response = requests.post(N8N_UPLOAD_URL, files=files_data, data=data)
        duree = round(time.time() - debut, 2)

    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Ingestion réussie en {duree}s")
        print(f"   Fichier    : {result.get('details', {}).get('fileName', nom_fichier)}")
        print(f"   Type       : {result.get('details', {}).get('fileType', 'N/A')}")
        print(f"   Mots       : {result.get('details', {}).get('wordCount', 'N/A')}")
        print(f"   Namespace  : {nom_fichier} (dans Pinecone)")
        return result
    else:
        print(f"❌ Erreur {response.status_code}: {response.text}")
        return None

# Upload le fichier depuis ton PC
print("\n📁 Sélectionne ton fichier cours...")
uploaded = files.upload()

# Récupère le nom du fichier uploadé
nom_fichier = list(uploaded.keys())[0]
print(f"\n✅ Fichier reçu : {nom_fichier}")

# Envoie vers n8n
result = upload_fichier(nom_fichier)

✅ Configuration chargée
   Index Pinecone : studybuddy-index
   Modèle embedding : Google Gemini Embeddings
   Taille des chunks : 500 tokens
   Overlap : 50 tokens

📁 Sélectionne ton fichier cours...


Saving chapitre3_IOT.pdf to chapitre3_IOT.pdf

✅ Fichier reçu : chapitre3_IOT.pdf
📤 Upload de : chapitre3_IOT.pdf
   Vers : https://sttudybuddy.app.n8n.cloud/webhook/rag-ingestion

✅ Ingestion réussie en 6.33s
   Fichier    : chapitre3_IOT.pdf
   Type       : PDF
   Mots       : 6537
   Namespace  : chapitre3_IOT.pdf (dans Pinecone)


## 4. Formats supportés et stratégies d'extraction

In [ ]:
formats_supportes = [
    {
        "format": "PDF",
        "extension": ".pdf",
        "outil_n8n": "extractFromFile (PDF)",
        "strategie": "Extraction texte natif PyPDF",
        "limite": "PDFs scannés → fallback OCR"
    },
    {
        "format": "PowerPoint",
        "extension": ".pptx",
        "outil_n8n": "extractFromFile (PPTX)",
        "strategie": "Extraction texte des slides",
        "limite": "Images dans slides non lues"
    },
    {
        "format": "Word",
        "extension": ".docx",
        "outil_n8n": "extractFromFile (DOCX)",
        "strategie": "Extraction texte python-docx",
        "limite": "Tableaux complexes peuvent perdre structure"
    },
    {
        "format": "Texte",
        "extension": ".txt",
        "outil_n8n": "extractFromFile (TXT)",
        "strategie": "Lecture directe UTF-8",
        "limite": "Aucune"
    },
    {
        "format": "Image JPG/PNG",
        "extension": ".jpg / .png",
        "outil_n8n": "Agent OCR (Gemini Vision)",
        "strategie": "Vision LLM → extraction texte",
        "limite": "Coût plus élevé, plus lent"
    }
]

print("📋 Formats supportés par la pipeline d'ingestion Study-Buddy")
print("=" * 70)
for fmt in formats_supportes:
    print(f"\n{'─'*50}")
    print(f"  Format     : {fmt['format']} ({fmt['extension']})")
    print(f"  Outil n8n  : {fmt['outil_n8n']}")
    print(f"  Stratégie  : {fmt['strategie']}")
    print(f"  Limite     : {fmt['limite']}")

📋 Formats supportés par la pipeline d'ingestion Study-Buddy

──────────────────────────────────────────────────
  Format     : PDF (.pdf)
  Outil n8n  : extractFromFile (PDF)
  Stratégie  : Extraction texte natif PyPDF
  Limite     : PDFs scannés → fallback OCR

──────────────────────────────────────────────────
  Format     : PowerPoint (.pptx)
  Outil n8n  : extractFromFile (PPTX)
  Stratégie  : Extraction texte des slides
  Limite     : Images dans slides non lues

──────────────────────────────────────────────────
  Format     : Word (.docx)
  Outil n8n  : extractFromFile (DOCX)
  Stratégie  : Extraction texte python-docx
  Limite     : Tableaux complexes peuvent perdre structure

──────────────────────────────────────────────────
  Format     : Texte (.txt)
  Outil n8n  : extractFromFile (TXT)
  Stratégie  : Lecture directe UTF-8
  Limite     : Aucune

──────────────────────────────────────────────────
  Format     : Image JPG/PNG (.jpg / .png)
  Outil n8n  : Agent OCR (Gemini Vis

## 5. Gestion de la Context Window

**Stratégie employée pour éviter l'explosion des tokens :**

- Le texte est **tronqué à 50 000 caractères** maximum avant chunking (node Nettoyer + Métadonnées)
- Le **chunking 500/50** garantit que chaque vecteur Pinecone reste sous la limite du modèle d'embedding
- Lors de la **recherche RAG**, seuls les **top-K chunks** (K=3 par défaut) sont injectés dans le prompt
- Le prompt de chaque agent est **limité à 2 phrases de réponse** pour éviter la sur-génération

In [ ]:
# Simulation de la stratégie de gestion context window
def analyser_texte(texte, limite_chars=50000, top_k=3):
    """
    Simule la gestion de la context window telle qu'implémentée dans n8n.
    """
    print("📊 Analyse de la gestion Context Window")
    print(f"   Texte original    : {len(texte)} caractères")

    # Étape 1: Troncature
    texte_tronque = texte[:limite_chars]
    print(f"   Après troncature  : {len(texte_tronque)} caractères (limite: {limite_chars})")

    # Étape 2: Chunking
    chunks = simulate_chunking(texte_tronque, chunk_size=500, overlap=50)
    print(f"   Chunks créés      : {len(chunks)}")

    # Étape 3: Recherche RAG (top-K)
    print(f"   Chunks injectés   : top {top_k} chunks pertinents")
    tokens_injectes = top_k * 500
    print(f"   Tokens max prompt : ~{tokens_injectes} tokens (RAG context)")
    print(f"   ✅ Context window respectée")

analyser_texte(exemple_texte * 10)

📊 Analyse de la gestion Context Window
   Texte original    : 3640 caractères
   Après troncature  : 3640 caractères (limite: 50000)
   Chunks créés      : 2
   Chunks injectés   : top 3 chunks pertinents
   Tokens max prompt : ~1500 tokens (RAG context)
   ✅ Context window respectée


## 6. Log d'ingestion — Preuve de fonctionnement

Voici un exemple de log réel retourné par le webhook après ingestion réussie :

```json
{
  "success": true,
  "message": "✅ Fichier ingéré avec succès !",
  "details": {
    "fileName": "chapitre3_IOT.pdf",
    "fileType": "PDF",
    "wordCount": 1842,
    "ingestedAt": "2026-05-16T21:05:00.000Z"
  }
}
```

**Namespace Pinecone créé :** `chapitre3_IOT.pdf`  
**Index :** `studybuddy-index`  
**Vecteurs stockés :** ~37 chunks × 768 dimensions (Gemini Embeddings)